In [ ]:
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
import os
import csv
import uuid

# Load the environment variables
load_dotenv()

chroma_client = chromadb.PersistentClient(path="./chroma_db")
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name="text-embedding-3-small"
)

job_collection = chroma_client.get_or_create_collection(
    name="job_postings",
    embedding_function=openai_ef
)

In [ ]:
# Prepare ids, docs, metadatas in an array to be inserted.

ids = []
documents = []
metadatas = []
with open("job_postings_with_metadata.csv", "r", encoding="utf-8") as infile:
    reader = csv.DictReader(infile)
    for idx, row in enumerate(reader):
        job_title = row.get("title", "").strip()
        job_content = row.get("content", "").strip()

        # Stop processing if row is empty (no more data)
        if not job_title or not job_content:
            print(f"Reached empty row at {idx}. Stopping processing.")
            break
        
        ids.append(str(uuid.uuid4()))
        documents.append(f"{job_title} {job_content[:2000]}")
        # location_type,location,salary_min,salary_max,salary_currency,department,job_type,industry
        metadata = {
            'url': row.get("absolute_url", "").strip(),
            'location_type': row.get("location_type", "").strip(),
            'location': row.get("location", "").strip(),  
            'salary_min': row.get("salary_min", "").strip(),
            'salary_max': row.get("salary_max", "").strip(),
            'salary_currency': row.get("salary_currency", "").strip(),
            'department': row.get("department", "").strip(),
            'job_type': row.get("job_type", "").strip(),
            'industry': row.get("industry", "").strip(),
        }
        metadatas.append(metadata)

In [ ]:
# Add to the vector db in batches

batch_size = 10
for i in range(0, len(ids), batch_size):
    job_collection.add(
        ids = ids[i:i + batch_size],
        documents = documents[i:i + batch_size],
        metadatas = metadatas[i:i + batch_size]
    )
    print(f"Added batch {i//batch_size + 1}")
print(f"Total jobs in DB: {job_collection.count()}")

In [ ]:
# chroma_client.delete_collection(name="job_collection")

In [ ]:
# Test query 1
results = job_collection.query(
    query_texts=["Remote job in the US"],
    n_results=1
)

print(results)
# for i in range(len(results['']))